In [215]:
import numpy as np
import matplotlib.pyplot as plt
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from scipy.integrate import solve_ivp

In [216]:
def Chaos_Function(t, y):
    x0 = y[0]
    dx0 = y[1]

    dx = dx0
    ddx = -delta*dx0 + x0 - x0**3 + F*np.cos(omega*t)

    return [dx, ddx]

In [217]:
def Potential(x):
    return (x**4)/4 -(x**2)/2

In [218]:
def Poincare_Plot(
        t,
        x,
        dx,
        n_start,
):

    T = 2* np.pi / omega
    n_end = int(t[-1]/ T)

    t_poincare = np.array([n * T for n in range(n_start, n_end)])

    # 보간해서 정확히 t = nT에서의 x, dx 값을 구함
    x_poincare = np.interp(t_poincare, t, x)
    dx_poincare = np.interp(t_poincare, t, dx)

    return x_poincare, dx_poincare

In [219]:
# 파라미터 조정
delta = 0.25
F = 0.40
omega = 1

#초기 값
x0 = 0
dx0 = -0.1
y0 = [x0, dx0]

#시간 지정
t_start = 0
t_end = 1000
t = np.linspace(t_start, t_end, 1000)

# 미분방정식 해 구하기
sol = solve_ivp(
    Chaos_Function,
    [t_start, t_end],
    y0,
    t_eval=t,
    method="RK45"
)

t = sol.t
x = sol.y[0]
dx = sol.y[1]

x_poincare, dx_poincare = Poincare_Plot(t, x, dx, n_start = 0)

z = Potential(x)
Z_poincare = Potential(x_poincare)

In [220]:
fig = make_subplots(
    rows = 4, cols = 1,
    subplot_titles=["Time - Position", "Time - Momentum", "Position - Momentum", "Position - Poincare"]
    )

fig.update_layout(width=1200,height=800)

fig.add_trace(
    go.Scatter(x = t, y = x),
    row=1, col=1,
)
fig.update_xaxes(title_text = "Time", row=1, col=1)
fig.update_yaxes(title_text = "Position", row=1, col=1)

fig.add_trace(
    go.Scatter(x = t, y = dx),
    row = 2, col = 1
)
fig.update_xaxes(title_text = "Time", row=2, col=1)
fig.update_yaxes(title_text = "Momentum", row=2, col=1)

fig.add_trace(
    go.Scatter(x = x, y = dx),
    row = 3, col = 1
)
fig.update_xaxes(title_text = "Position", row=3, col=1)
fig.update_yaxes(title_text = "Momentum", row=3, col=1)

fig.add_trace(
    go.Scatter(x = x_poincare, y = dx_poincare, mode="markers"),
    row = 4, col = 1,
    
)
fig.update_xaxes(title_text = "Poincare Position", row=4, col=1)
fig.update_yaxes(title_text = "Poincare Momentum", row=4, col=1)

In [221]:
def Fig3d(x_data, y_data, z_data, axes_label, title, mode = "lines"):
    fig3d = go.Figure(go.Scatter3d(
    x = x_data,
    y = y_data,
    z = z_data,
    mode=mode,
    line = dict(width = 2),
    marker = dict(size = 2)
        )
    )

    fig3d.update_layout(
        scene=dict(
            xaxis_title=axes_label[0],
            yaxis_title=axes_label[1],
            zaxis_title=axes_label[2]
        ),
        width=800,
        height=700,
        title = title
    )

    fig3d.show()

In [222]:
axes_label = [
    ["Position", "Momentum", "Potential"],
    ["Time", "Position", "Potential"],
    ["Time", "Momentum", "Potential"],
    ["Poincare Position", "Poincare Momentum", "Poincare Potential"],
    ["Position", "Potentioal", "Poincare Potential"]
    ]

title = ["Position - Momentum - Potential",
        "Time - Position - Potential",
        "Time - Momentum - Potential",
        "Poincare Position - Poincare Momentum - Poincare Potential",
        "Position - Potentioal - Poincare Potentioal"
        ]

Fig3d(x, dx, z, axes_label[0], title[0])
Fig3d(t, x, z, axes_label[1], title[1])
Fig3d(t, dx, z, axes_label[2], title[2])
Fig3d(x_poincare, dx_poincare, Z_poincare, axes_label[3], title[3])
Fig3d(x, z, Z_poincare, axes_label[4], title[4], "markers")